In [3]:
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
from openpiv import filters, pyprocess, validation


# Parametri da adattare ai tuoi dati
INPUT_DIR = Path("F:\\Luca_Data\\20260925_test_microscope_and_camera\\fluoromax_0.25_1000fps_1uls")
OUTPUT_DIR = Path("risultati\\piv")
PATTERN = "*.tif"

WINDOW_SIZE = 32
OVERLAP = 16
SEARCH_AREA_SIZE = 64
DT_SECONDS = 1/1000        # tempo reale tra i frame confrontati (S)
S2N_THRESHOLD = 1.3        # soglia del rapporto segnale/rumore

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def read_gray(path):
    image = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    if image is None:
        raise ValueError(f"Impossibile leggere l'immagine: {path}")
    return image


def analyze_pair(path_a, path_b):
    frame_a = read_gray(path_a)
    frame_b = read_gray(path_b)

    if frame_a.shape != frame_b.shape:
        raise ValueError(f"Dimensioni diverse: {path_a.name}, {path_b.name}")

    # Stima degli spostamenti; dt è l'intervallo temporale tra i due frame
    u, v, s2n = pyprocess.extended_search_area_piv(
        frame_a,
        frame_b,
        window_size=WINDOW_SIZE,
        overlap=OVERLAP,
        search_area_size=SEARCH_AREA_SIZE,
        dt=DT_SECONDS,
        sig2noise_method="peak2peak",
    )

    x, y = pyprocess.get_coordinates(
        image_size=frame_a.shape,
        search_area_size=SEARCH_AREA_SIZE,
        overlap=OVERLAP,
    )

    # Filtra i vettori poco affidabili e interpola gli outlier
    u, v, invalid = validation.sig2noise_val(
        u, v, s2n, threshold=S2N_THRESHOLD
    )
    u, v = filters.replace_outliers(
        u, v, invalid,
        method="localmean",
        max_iter=3,
        kernel_size=3,
    )

    return x, y, u, v, s2n, invalid


def main():
    # I nomi zero-padded, come frame_0001.tif, frame_0002.tif,
    # evitano ordinamenti lessicografici errati.
    frames = sorted(INPUT_DIR.glob(PATTERN))
    if len(frames) < 2:
        raise RuntimeError(f"Servono almeno due immagini in {INPUT_DIR}")

    for index, (path_a, path_b) in enumerate(zip(frames, frames[1:])):
        x, y, u, v, s2n, invalid = analyze_pair(path_a, path_b)

        result = pd.DataFrame({
            "x_px": x.ravel(),
            "y_px": y.ravel(),
            "u_px_s": u.ravel(),
            "v_px_s": v.ravel(),
            "signal_to_noise": s2n.ravel(),
            "invalid": invalid.ravel(),
        })

        output_path = OUTPUT_DIR / f"piv_{index:04d}.csv"
        result.to_csv(output_path, index=False)
        print(f"Salvato: {output_path.name}")


if __name__ == "__main__":
    main()

TypeError: sig2noise_val() got multiple values for argument 'threshold'